# 🐾 CHALLENGE: Agente de una Veterinaria con Llama Stack

Notebook de respaldo del proyecto **`Agente_Veterinaria_LlamaStack`**.

Sigue la estructura del hands-on de la masterclass *Diseño de Agentes Conversacionales*, con el reto de la veterinaria:

1. **Entorno**: Ollama + `llama3.1:8b` + Llama Stack en Colab, credenciales en Colab Secrets.
2. **Memoria**: una conversación de Llama Stack por número de teléfono (`client.conversations.create()`).
3. **Reglas fijas**: escalamiento a persona **y urgencia médica** + lock por número.
4. **Webhook**: endpoints de verificación y recepción con `BackgroundTasks`.
5. **Prueba**: 4+ mensajes con historial recortado + palabra de urgencia que dispara la regla.

> **Versión local (PC):** la misma práctica ejecutada con Ollama local y `gemma4:12b`
> está en el repositorio, en `src/` con sus resultados en `resultados/`.
> Aquí se usa **Llama 3.1 8B** tal como lo pide el challenge.

## 1. Entorno

### Colab Secrets

Guarda el token de WhatsApp en el panel de **Secrets** (ícono de llave) con el nombre
`WHATSAPP_TOKEN` y activa *Notebook access*. No pegues la key en el código.

In [ ]:
# Leer credenciales desde Colab Secrets
from google.colab import userdata

WHATSAPP_TOKEN = userdata.get('WHATSAPP_TOKEN')
VERIFY_TOKEN = 'VET_2026'
PHONE_ID = userdata.get('WHATSAPP_PHONE_ID', '')
print('Secrets cargados (token oculto):', bool(WHATSAPP_TOKEN))

### Instalar Ollama en Colab

Colab no trae Ollama; se instala y arranca en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama (Colab no trae zstd por defecto)
!sudo apt-get -y install zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nohup ollama serve > /tmp/ollama.log 2>&1 &
import time; time.sleep(5)
!curl -s http://127.0.0.1:11434/api/version

In [ ]:
# Descargar Llama 3.1 8B (el modelo del challenge)
!ollama pull llama3.1:8b
!ollama list

### Instalar y levantar Llama Stack

Llama Stack estandariza la pieza de **memoria** (conversations), **herramientas** y
**seguridad** para no reconstruirla en cada proyecto.

> Nota de versión: con `llama-stack>=0.5` el servidor se arranca con
> `llama stack run <run.yaml>` (el comando `llama-stack serve` ya no existe).

In [ ]:
# Instalar dependencias + Llama Stack
!pip -q install "llama-stack==0.7.3" "llama-stack-client==0.7.5" "mcp<1.13" "pydantic<2.12" ollama fastapi "uvicorn[standard]" requests pyngrok

In [ ]:
%%writefile run.yaml
# Distribucion minima: Ollama local (o el Ollama de Colab) como inferencia.
version: 2
image_name: veterinary-agent
apis:
  - inference
  - responses
  - vector_io
  - tool_runtime
  - files
providers:
  inference:
    - provider_id: ollama
      provider_type: remote::ollama
      config:
        base_url: http://127.0.0.1:11434/v1
  responses:
    - provider_id: builtin
      provider_type: inline::builtin
      config:
        persistence:
          agent_state: {namespace: agents, backend: kv_default}
          responses: {table_name: responses, backend: sql_default}
  vector_io:
    - provider_id: sqlite-vec
      provider_type: inline::sqlite-vec
      config:
        db_path: data/sqlite_vec.db
        persistence: {namespace: vector_io::sqlite_vec, backend: kv_default}
  tool_runtime:
    - provider_id: file-search
      provider_type: inline::file-search
  files:
    - provider_id: builtin-files
      provider_type: inline::localfs
      config:
        storage_dir: data/files
        metadata_store: {table_name: files_metadata, backend: sql_default}
storage:
  backends:
    kv_default: {type: kv_sqlite, db_path: data/kvstore.db}
    sql_default: {type: sql_sqlite, db_path: data/sql_store.db}
  stores:
    metadata: {namespace: registry, backend: kv_default}
    inference: {table_name: inference_store, backend: sql_default}
    conversations: {table_name: openai_conversations, backend: sql_default}
    prompts: {namespace: prompts, backend: kv_default}
    connectors: {namespace: connectors, backend: kv_default}
registered_resources:
  models:
    - {model_id: llama3.1:8b, provider_id: ollama, provider_model_id: llama3.1:8b, model_type: llm}
  shields: []
  vector_stores: []
  datasets: []
  scoring_fns: []
  benchmarks: []

In [ ]:
# Levantar Llama Stack en segundo plano y esperar el health
import subprocess, time, requests

proceso = subprocess.Popen(
    ['llama', 'stack', 'run', 'run.yaml', '--port', '8321'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(60):
    time.sleep(2)
    try:
        if requests.get('http://127.0.0.1:8321/v1/health', timeout=3).ok:
            print('Llama Stack listo en :8321')
            break
    except Exception:
        pass
else:
    raise RuntimeError('Llama Stack no arrancó; revisa run.yaml')

from llama_stack_client import LlamaStackClient
client = LlamaStackClient(base_url='http://127.0.0.1:8321')
print('modelos:', [m.identifier for m in client.models.list()])

## 2. Lógica del asistente

### Memoria por número de teléfono

Sin esto, Llama trataría cada mensaje como si fuera el primero. Cada número tiene
su propia **conversación de Llama Stack** y un **lock** para serializar sus mensajes.

In [ ]:
%%writefile agente_veterinaria.py
"""Agente veterinario: memoria por telefono en Llama Stack + reglas fijas."""
import threading
from llama_stack_client import LlamaStackClient

client = LlamaStackClient(base_url='http://127.0.0.1:8321')
MODELO = 'ollama/llama3.1:8b'
MAX_ITEMS = 8          # limite de historial (challenge)
INSTRUCCIONES = (
    'Eres el asistente de la clinica veterinaria Patitas Felices. '
    'Responde en espanol, amable, en maximo 3 lineas. No inventes '
    'horarios, precios ni diagnosticos.'
)

# --- Sesiones por telefono: conversacion + lock + contadores ---
sesiones = {}           # telefono -> dict
sesiones_lock = threading.Lock()

def sesion(telefono: str) -> dict:
    with sesiones_lock:
        if telefono not in sesiones:
            conv = client.conversations.create(
                metadata={'telefono': telefono, 'proyecto': 'veterinaria'})
            sesiones[telefono] = {
                'conversation_id': conv.id,
                'lock': threading.Lock(),   # lock POR NUMERO
                'turnos': 0, 'borrados': 0,
            }
        return sesiones[telefono]

def items(conversation_id: str):
    """Historial en orden cronologico (asc)."""
    pag = client.conversations.items.list(conversation_id=conversation_id, order='asc')
    return list(pag.data)

def recortar(s: dict):
    """Conserva solo los MAX_ITEMS mas recientes (challenge: limite al historial)."""
    lista = items(s['conversation_id'])
    exceso = len(lista) - MAX_ITEMS
    borrados = 0
    for it in lista[:max(exceso, 0)]:          # los MAS ANTIGUOS
        client.conversations.items.delete(
            item_id=it.id, conversation_id=s['conversation_id'])
        borrados += 1
    s['borrados'] += borrados
    return {'antes': len(lista), 'borrados': borrados,
            'despues': len(lista) - borrados}

### Reglas fijas: escalamiento y urgencia médica

Dos decisiones que **no se delegan al modelo**:

- **Escalamiento**: el usuario pide hablar con una persona → mensaje fijo.
- **Urgencia médica** (challenge): palabra de emergencia → respuesta inmediata,
  una mascota en urgencia no puede esperar a que el modelo decida.

Se evalúan **antes** de invocar a Llama: si disparan, la conversación no crece.

In [ ]:
%%writefile reglas.py
"""Reglas fijas: escalamiento a persona y urgencia medica."""
import re, unicodedata

REGLAS = {
    'escalado_humano': {
        'patrones': ['quiero hablar con', 'hablar con un humano',
                     'hablar con una persona', 'quiero un asesor',
                     'me quiero quejar', 'un ejecutivo'],
        'mensaje': ('Claro, te paso con una persona del equipo veterinario. '
                    'Ya registre tu solicitud y te contactamos en breve.'),
    },
    'urgencia_medica': {
        'patrones': ['urgencia', 'urgente', 'emergencia', 'no respira',
                     'se desmaya', 'sangra mucho', 'sangrando', 'veneno',
                     'enveneno', 'atropellado', 'convulsiones'],
        'mensaje': ('URGENCIA: tu mascota necesita atencion inmediata. '
                    'Llevala ahora a la clinica mas cercana; no esperes '
                    'respuesta por aqui.'),
    },
}

def _normalizar(t: str) -> str:
    t = unicodedata.normalize('NFD', (t or '').lower())
    t = ''.join(c for c in t if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^a-z0-9\s]', ' ', t)

def evaluar(texto: str):
    """Devuelve (nombre_regla, mensaje) o (None, None)."""
    limpio = _normalizar(texto)
    for nombre, regla in REGLAS.items():
        if any(_normalizar(p) in limpio for p in regla['patrones']):
            return nombre, regla['mensaje']
    return None, None

In [ ]:
%%writefile flujo.py
"""Procesamiento: lock por numero -> reglas fijas -> Llama Stack -> recorte."""
from agente_veterinaria import sesion, items, recortar, client, MODELO, INSTRUCCIONES
from reglas import evaluar

def procesar(telefono: str, mensaje: str) -> dict:
    s = sesion(telefono)
    with s['lock']:                       # serializa mensajes del mismo numero
        # 1) Reglas fijas primero (sin modelo)
        nombre, respuesta = evaluar(mensaje)
        if nombre:
            return {'fuente': 'regla', 'regla': nombre,
                    'respuesta': respuesta}
        # 2) Modelo con memoria (la conversacion de Llama Stack)
        r = client.responses.create(
            model=MODELO, conversation=s['conversation_id'],
            instructions=INSTRUCCIONES,
            input=[{'role': 'user', 'content': mensaje}],
            max_output_tokens=180, temperature=0.3,
        )
        # 3) Limite de historial
        recorte = recortar(s)
        s['turnos'] += 1
        return {'fuente': 'llama-stack', 'respuesta': (r.output_text or '').strip(),
                'recorte': recorte, 'turno': s['turnos']}

## 3. Despliegue del webhook

Los dos endpoints que WhatsApp necesita: verificación y recepción.
`BackgroundTasks` responde a Meta de inmediato y delega el procesamiento
real (evita reintentos y respuestas duplicadas).

In [ ]:
%%writefile webhook.py
"""Webhook de WhatsApp con BackgroundTasks (FastAPI)."""
import re, requests
from fastapi import BackgroundTasks, FastAPI, Query, Request
from flujo import procesar

app = FastAPI()
VERIFY_TOKEN = 'VET_2026'
TOKEN, PHONE_ID = '', ''   # se llenan en runtime desde Colab Secrets
procesados = set()         # ids ya atendidos (evita duplicados)

def normalizar_numero(tel: str) -> str:
    n = re.sub(r'[^0-9+]', '', tel or '')
    n = n[1:] if n.startswith('+') else n
    n = n[3:] if n.startswith(('044', '045')) else n
    if len(n) == 10: n = '52' + n
    return n

@app.get('/webhook')
def verificar(hub_mode=Query('', alias='hub.mode'),
              hub_verify_token=Query('', alias='hub.verify_token'),
              hub_challenge=Query('', alias='hub.challenge')):
    if hub_mode == 'subscribe' and hub_verify_token == VERIFY_TOKEN:
        return int(hub_challenge) if hub_challenge.isdigit() else hub_challenge
    return {'error': 'token invalido'}

def enviar_respuesta(to: str, texto: str, contexto: str):
    requests.post(
        f'https://graph.facebook.com/v19.0/{PHONE_ID}/messages',
        headers={'Authorization': f'Bearer {TOKEN}'},
        json={'messaging_product': 'whatsapp', 'to': to,
              'type': 'text', 'text': {'body': texto},
              'context': {'message_id': contexto}}, timeout=30)

def procesar_y_responder(wamid, telefono, texto):
    r = procesar(telefono, texto)
    enviar_respuesta(telefono, r['respuesta'], wamid)

@app.post('/webhook')
async def recibir(request: Request, background: BackgroundTasks):
    cuerpo = await request.json()
    for entrada in cuerpo.get('entry', []):
        for cambio in entrada.get('changes', []):
            valor = cambio.get('value', {})
            for m in valor.get('messages', []) or []:
                if m.get('type') != 'text':
                    continue
                wamid = m.get('id', '')
                if wamid in procesados:
                    continue
                procesados.add(wamid)
                background.add_task(
                    procesar_y_responder, wamid,
                    normalizar_numero(m.get('from', '')),
                    m.get('text', {}).get('body', ''))
    return {'status': 'received'}

In [ ]:
# Arrancar el servidor en un hilo aparte
import threading, uvicorn
from webhook import app
TOKEN, PHONE_ID = WHATSAPP_TOKEN, PHONE_ID
import webhook as _wh
_wh.TOKEN, _wh.PHONE_ID = WHATSAPP_TOKEN, PHONE_ID

def correr():
    uvicorn.run(app, host='0.0.0.0', port=8020, log_level='warning')

threading.Thread(target=correr, daemon=True).start()
import time; time.sleep(3)
print('Webhook en :8020')

## 4. Túnel con ngrok

La URL impresa (`https://…ngrok.io/webhook`) es la que se configura como
*Callback URL* en **Meta for Developers** (WhatsApp > Configuración de la API >
Configurar webhooks).

In [ ]:
from pyngrok import ngrok
tunel = ngrok.connect(8020, 'http')
print('Callback URL:', f'{tunel.public_url}/webhook')

## 5. Probar el flujo completo

Prueba local **sin esperar a WhatsApp** (misma lógica que el webhook, directa
sobre el agente): 4 mensajes donde el 3º y 4º dependen del primero, y un
mensaje de urgencia. Se verifica que el historial se recorta y que la regla
dispara sin pasar por el modelo.

In [ ]:
from flujo import procesar
from agente_veterinaria import sesiones, items

TEL = '5215511110009'
mensajes = [
    'Hola, quiero una cita dental para mi perrito Max',  # 1: menciona servicio
    'El martes me queda mejor',                          # 2: depende del 1
    'Son las 10 de la manana?',                          # 3: depende del 1 y 2
    'Perfecto, agendala',                                # 4: cierra sin repetir servicio
    'Cual fue el servicio que pedi?',                    # 5: prueba de memoria + recorte
]
for txt in mensajes:
    r = procesar(TEL, txt)
    extra = ''
    if r['fuente'] == 'regla':
        extra = f"  [REGLA {r['regla']}]"
    elif r.get('recorte', {}).get('borrados'):
        extra = f"  [recorte: -{r['recorte']['borrados']} items]"
    print(f'> {txt}')
    print(f"  {r['respuesta'][:120]}{extra}\n")

In [ ]:
# Palabra de urgencia: debe disparar la REGLA, sin invocar al modelo
r = procesar(TEL, 'Mi perrito esta sangrando mucho, es urgente!')
print('fuente :', r['fuente'], '| regla:', r.get('regla'))
print('respuesta:', r['respuesta'])

# Escalado a humano: tambien regla fija
r2 = procesar(TEL, 'Quiero hablar con una persona')
print('fuente :', r2['fuente'], '| regla:', r2.get('regla'))

# Ver historial recortado en Llama Stack
s = sesiones[TEL]
lista = items(s['conversation_id'])
print(f"\nhistorial conservado: {len(lista)} items (limite 8) | "
      f"borrados en total: {s['borrados']}")
for it in lista[-4:]:
    cont = it.content if isinstance(it.content, str) else (
        next((p.text for p in it.content if getattr(p, 'text', None)), ''))
    print(f'  [{getattr(it, "role", "")}] {cont[:80]}')

### Cerrar el túnel

Al terminar la sesión:

In [ ]:
ngrok.disconnect(tunel.public_url)
print('Tunel cerrado.')

---

**Resumen del challenge:**

| Requisito | Implementación |
|---|---|
| Conversación por número | `client.conversations.create(metadata={'telefono': …})` |
| Regla de escalamiento | `reglas.escalado_humano` (antes del modelo) |
| Regla de urgencia médica (challenge) | `reglas.urgencia_medica` (antes del modelo) |
| Límite al historial (challenge) | `MAX_ITEMS = 8` + `recortar()` con `order='asc'` |
| Lock por número (challenge) | `threading.Lock` dentro de cada sesión |
| Webhook con BackgroundTasks | `POST /webhook` responde 200 y procesa en fondo |

Versión local con resultados ejecutados: ver `resultados/` y `docs/EJERCICIOS.md` del repositorio.